export CAC_API_KEY="API KEY GOES HERE" # Run this in terminal or Use .env file and add it to .gitignore

In [5]:
import pandas as pd
import requests
import os
from openai import OpenAI
# Environment variable for API key
# api_key = os.getenv("CAC_API_KEY")

# .env file
from dotenv import load_dotenv
load_dotenv()  # loads .env
api_key = os.getenv("CAC_API_KEY")

host = "https://maki.uni-mannheim.de/v1"
model = "ministral-3-14b"


In [6]:
df = pd.read_csv("dataset/all_multi_paragraphs_2022_2026_translated.csv")

test_df = df.tail(5)
test_df

,article_id,pub,par_index,text_block,group,length,text_block_en
29995,BADZ-51309103090,BADZ,5,"Er wolle nach Italien, sagt einer aus der Rund...",REFUG,194,"He wants to go to Italy, says one from the gro..."
29996,BADZ-51309103090,BADZ,7,"Die Personen wurden kontrolliert, kamen sie oh...",REFUG,220,"The people were checked, if they came without ..."
29997,BADZ-51309103090,BADZ,10,In einem Aktionsplan wurde eine engere Zusamme...,REFUG,223,"In an action plan, a closer cooperation betwee..."
29998,BADZ-51309103090,BADZ,16,Wie die SBB wehren sich auch die Grenzwache un...,REFUG,227,How the SBB defend themselves also do the bord...
29999,BADZ-51309103090,BADZ,20,"Das Prozedere, die [Andere Gruppe] durchreisen...",REFUG,215,The procedure of allowing [andere Gruppe] to p...


In [3]:
# Few-shot examples
example_instructions_expl = "Please classify each paragraph according to the given labels and explain why."
example_fewshot = {
    "input": [
        "This is the first example paragraph.",
        "This is the second example paragraph."
    ],
    "explanation": [
        "This paragraph talks about topic A, so label is X.",
        "This paragraph mentions topic B, so label is Y."
    ],
    "label": ["X", "Y"]
}
example_reminder_expl = "Always provide the reasoning before the final label."

### API TEST

In [7]:
# Function to construct few-shot prompt for each paragraph
def generate_prompt(paragraph):
    prompt = "Instructions: " + example_instructions_expl + "\n\n"
    for inp, expl, label in zip(example_fewshot["input"], example_fewshot["explanation"], example_fewshot["label"]):
        prompt += f"Example:\nInput: {inp}\nExplanation: {expl}\nLabel: {label}\n\n"
    prompt += example_reminder_expl + "\n\n"
    prompt += "Now classify or explain this paragraph:\n" + paragraph
    return prompt

# Loop through the dataset and call API
results = []

for idx, row in test_df.iterrows():
    paragraph = row['text_block']
    article_id = row['article_id']
    
    prompt = generate_prompt(paragraph)
    
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": prompt}
        ]
    }
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    url = f"{host}/chat/completions"
    response = requests.post(url, json=payload, headers=headers)
    
    if response.status_code == 200:
        content = response.json()['choices'][0]['message']['content']
        results.append({
            "article_id": article_id,
            "par_index": row['par_index'],
            "text_block": paragraph,
            "model_output": content
        })
    else:
        print(f"Error with article {article_id}, paragraph {row['par_index']}. Status code: {response.status_code}")
        results.append({
            "article_id": article_id,
            "par_index": row['par_index'],
            "text_block": paragraph,
            "model_output": None
        })


In [8]:
# Convert results to DataFrame
results_df = pd.DataFrame(results)
print(results_df)

         article_id  par_index  \
0  BADZ-51309103090          5   
1  BADZ-51309103090          7   
2  BADZ-51309103090         10   
3  BADZ-51309103090         16   
4  BADZ-51309103090         20   

                                          text_block  \
0  Er wolle nach Italien, sagt einer aus der Rund...   
1  Die Personen wurden kontrolliert, kamen sie oh...   
2  In einem Aktionsplan wurde eine engere Zusamme...   
3  Wie die SBB wehren sich auch die Grenzwache un...   
4  Das Prozedere, die [Andere Gruppe] durchreisen...   

                                        model_output  
0  Here’s the classification with reasoning for e...  
1  ### **Textausschnitt-Analyse**\n\n**Relevante ...  
2  **Explanation:**\nThis paragraph addresses the...  
3  Here’s the classification and explanation of t...  
4  Here’s the classification of the paragraph bas...  
